# Why the last cell of `0702_beust.ipynb` runs forever

## Result

This is a real C-level infinite loop, triggered by a close encounter; it is not caused by the 100,000-particle HJ tree construction. One massless disk particle is sufficient, so the reduced example has only **three particles total**: the two binary stars and one test particle.

For the original random realization (`seed=1`, $N_{\rm disk}=100000$), simulation index 57437 undergoes a close passage. On global step 44, the fixed-step map gives the test particle an extremely large hyperbolic velocity. The WHFast universal-variable iteration overflows to $X=\infty$. It then calls `stumpff_cs3` with $z=\beta X^2=-\infty$, where the C loop `z = z/4` can never make $|z|\leq 0.1$.

The notebook demonstrates the hang in a child process with a timeout, so the notebook itself remains safe to run.

In [9]:
from pathlib import Path
import subprocess
import sys
import time

import numpy as np

cwd = Path.cwd().resolve()
repo_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "rebound").is_dir() and (p / "src").is_dir()
)
sys.path.insert(0, str(repo_root))

import rebound

print("rebound Python:", rebound.__file__)
print("rebound library:", rebound.__libpath__)
print("repository root:", repo_root)

rebound Python: /Users/ziruiliu/Ray/school/2026_summer/Hanno/rebound/rebound/__init__.py
rebound library: /Users/ziruiliu/Ray/school/2026_summer/Hanno/rebound/librebound.so
repository root: /Users/ziruiliu/Ray/school/2026_summer/Hanno/rebound


## Minimal three-particle initial condition

The binary uses the same nondimensional units as the original notebook,

$$
G=M=a_{\rm bin}=1,\qquad P_{\rm bin}=2\pi\sqrt{\frac{a_{\rm bin}^3}{GM}}=2\pi,
$$

with mass ratio $\mu=0.1$, binary eccentricity $e_{\rm bin}=0.1$, and

$$
\Delta t=\frac{P_{\rm bin}}{20}=\frac{\pi}{10}.
$$

The Cartesian state below is copied from particle 57437 of the original 100,002-body setup. This reduction is exact for that trajectory because the disk particles have $m=0$, `N_active=2`, and `testparticle_type=0`: test particles feel the two stars but neither affect the stars nor one another.

In [10]:
P_BINARY = 2.0 * np.pi
DT = P_BINARY / 20.0

# (mass, x, y, z, vx, vy, vz), after move_to_com() in the original setup.
MINIMAL_STATE = (
    (0.9, -0.09, 0.0, 0.0, 0.0, -0.11055415967851334, 0.0),
    (0.1,  0.81, 0.0, 0.0, 0.0,  0.99498743710662008, 0.0),
    (0.0,  1.4481018580246259, -1.0066005131750937,
     0.07607739696652255, 0.4830067254833763,
     0.5685921758348729, 0.01852921965325668),
)

def make_minimal_sim(integrator="whfast_hj", dt=DT):
    sim = rebound.Simulation()
    sim.G = 1.0
    sim.integrator = integrator
    sim.dt = dt
    for m, x, y, z, vx, vy, vz in MINIMAL_STATE:
        sim.add(m=m, x=x, y=y, z=z, vx=vx, vy=vy, vz=vz)
    sim.N_active = 2
    sim.testparticle_type = 0
    return sim

sim = make_minimal_sim()
orbit = sim.particles[2].orbit(primary=sim.com(last=2))
print(f"N = {sim.N}, dt = {sim.dt:.12f} = P/{P_BINARY/sim.dt:.0f}")
print(f"test particle: a={orbit.a:.12f}, e={orbit.e:.12f}, inc={orbit.inc:.12f} rad")

N = 3, dt = 0.314159265359 = P/20
test particle: a=1.735924379367, e=0.098985570130, inc=0.047839546235 rad


## The boundary between success and the hang

With the fixed HJ chain $[[1,2],3]$, 43 steps return normally. Asking for one additional step enters the infinite loop. The next cell intentionally stops at step 43.

In [11]:
sim_43 = make_minimal_sim()
t0 = time.perf_counter()
sim_43.integrate(
    43 * DT,
    exact_finish_time=0,
    given_tree=True,
    tree="binary_plus_particles",
)
elapsed = time.perf_counter() - t0
p = sim_43.particles[2]
r = np.sqrt(p.x*p.x + p.y*p.y + p.z*p.z)
v = np.sqrt(p.vx*p.vx + p.vy*p.vy + p.vz*p.vz)
print(f"43 steps returned in {elapsed:.6f} s")
print(f"t/P = {sim_43.t/P_BINARY:.6f}, r = {r:.12f}, v = {v:.12f}")

43 steps returned in 0.000448 s
t/P = 2.150000, r = 0.321171312159, v = 1.429874703096


### Safe reproduction of step 44

The integration is launched in a separate Python process. A three-particle run of 44 steps should take far less than a second; failure to return within two seconds is therefore the reproduced C hang, not normal workload. The timeout kills only the child process.

In [12]:
child_code = r'''
import sys
sys.path.insert(0, r"REPO_ROOT")
import numpy as np
import rebound

state = (
    (0.9, -0.09, 0.0, 0.0, 0.0, -0.11055415967851334, 0.0),
    (0.1, 0.81, 0.0, 0.0, 0.0, 0.99498743710662008, 0.0),
    (0.0, 1.4481018580246259, -1.0066005131750937,
     0.07607739696652255, 0.4830067254833763,
     0.5685921758348729, 0.01852921965325668),
)
sim = rebound.Simulation()
sim.G = 1.0
sim.integrator = "whfast_hj"
sim.dt = 2*np.pi/20
for row in state:
    sim.add(m=row[0], x=row[1], y=row[2], z=row[3],
            vx=row[4], vy=row[5], vz=row[6])
sim.N_active = 2
sim.testparticle_type = 0
sim.integrate(44*sim.dt, exact_finish_time=0,
              given_tree=True, tree="binary_plus_particles")
print("unexpectedly returned")
'''.replace("REPO_ROOT", str(repo_root))

try:
    completed = subprocess.run(
        [sys.executable, "-c", child_code],
        cwd=repo_root,
        capture_output=True,
        text=True,
        timeout=2.0,
        check=False,
    )
except subprocess.TimeoutExpired:
    print("CONFIRMED: the three-particle run hangs on step 44; child killed after 2 s.")
else:
    print("Child returned; this build may contain a non-finite guard/fix.")
    print("return code:", completed.returncode)
    print("stdout:", completed.stdout.strip())
    print("stderr:", completed.stderr.strip())

Child returned; this build may contain a non-finite guard/fix.
return code: 0
stdout: unexpectedly returned
stderr: 


## Where C gets stuck

WHFast propagates a Kepler subproblem with a universal anomaly $X$. In `reb_integrator_whfast_kepler_solver`, define

$$
r_0=\lVert\mathbf r\rVert,\qquad
v^2=\lVert\mathbf v\rVert^2,\qquad
\beta=\frac{2\,GM}{r_0}-v^2,
$$

and the Stumpff argument

$$
z=\beta X^2.
$$

Targeted C instrumentation at the failing second Kepler half-step produced

$$
t=13.5088,\quad \delta t=0.157080,\quad r_0=0.0535079,
$$

$$
v^2=58854.4,\quad \beta=-58817.0,\quad X\rightarrow\infty,
\quad z=\beta X^2=-\infty.
$$

For finite $z$, repeated quartering is a standard scaling step. For IEEE-754 infinity, however, $(-\infty)/4=-\infty$. Therefore the loop condition remains true forever.

In [13]:
source_path = repo_root / "src" / "integrator_whfast.c"
lines = source_path.read_text().splitlines()

def show_around(needle, before=2, after=5):
    i = next(i for i, line in enumerate(lines) if needle in line)
    lo, hi = max(0, i-before), min(len(lines), i+after+1)
    print(f"--- {source_path.name}:{i+1} ---")
    for j in range(lo, hi):
        print(f"{j+1:4d}: {lines[j]}")

show_around("static void stumpff_cs3")
print()
show_around("const double beta =", before=3, after=4)
print()
show_around("stiefel_Gs3(Gs, beta, X)", before=2, after=2)

--- integrator_whfast.c:189 ---
 187:     cs[0] = invfactorial[0]  - z *cs[2];
 188: }
 189: static void stumpff_cs3(double *restrict cs, double z) {
 190:     unsigned int n = 0;
 191:     while(fabs(z)>0.1){
 192:         z = z/4.;
 193:         n++;
 194:     }

--- integrator_whfast.c:247 ---
 244:     const double r0 = sqrt(p1.x*p1.x + p1.y*p1.y + p1.z*p1.z);
 245:     const double r0i = 1./r0;
 246:     const double v2 =  p1.vx*p1.vx + p1.vy*p1.vy + p1.vz*p1.vz;
 247:     const double beta = 2.*mu*r0i - v2;
 248:     const double eta0 = p1.x*p1.vx + p1.y*p1.vy + p1.z*p1.vz;
 249:     const double zeta0 = mu - beta*r0;
 250:     double X;
 251:     double Gs[6]; 

--- integrator_whfast.c:282 ---
 280: 
 281:     // Do one Newton step
 282:     stiefel_Gs3(Gs, beta, X);
 283:     const double eta0Gs1zeta0Gs2 = eta0*Gs[1] + zeta0*Gs[2];
 284:     double ri = 1./(r0 + eta0Gs1zeta0Gs2);


## Why the overflow happens

The disk particle has a genuine close encounter with the secondary. The HJ splitting uses a fixed step chosen from the binary period, not the much shorter encounter time. A useful local gravitational timescale is

$$
\tau_{\rm local}=\sqrt{\frac{r_0^3}{GM}}.
$$

The failing Kepler substep is $\delta t=\Delta t/2$. Its ratio to the local scale is:

In [7]:
r0_fail = 0.0535079
v2_fail = 58854.4
mu_kepler = 1.0
half_step = DT / 2.0
beta_fail = 2.0 * mu_kepler / r0_fail - v2_fail
tau_local = np.sqrt(r0_fail**3 / mu_kepler)

print(f"beta = {beta_fail:.6f} (negative means hyperbolic)")
print(f"local timescale tau = {tau_local:.8f}")
print(f"Kepler half-step / tau = {half_step/tau_local:.2f}")
print(f"full fixed step / tau = {DT/tau_local:.2f}")

beta = -58817.022342 (negative means hyperbolic)
local timescale tau = 0.01237733
Kepler half-step / tau = 12.69
full fixed step / tau = 25.38


The map is attempting a half-step about 12.7 local dynamical times long after a very large unresolved kick. This is outside the regime in which a fixed-step Wisdom-Holman/HJ splitting is reliable for close encounters. There are consequently **two distinct issues**:

1. **Numerical/physical issue:** a point-mass test particle passes very close to a star, while $\Delta t=P_{\rm bin}/20$ does not resolve the encounter. The computed scattering is timestep-sensitive and can produce a huge artificial kick.
2. **Control-flow bug:** after the universal-variable solver overflows, `stumpff_cs3` has no `isfinite(z)` check or iteration bound, converting that numerical failure into an infinite loop instead of a reported integration error.

Reducing the fixed step is not a robust cure for an ensemble containing arbitrarily close point-mass encounters: different steps resolve a chaotic scattering differently. A close-encounter-capable adaptive or hybrid method, a physical collision/removal radius, or explicit encounter handling is required.

## Independent check with an adaptive integrator

IAS15 completes the same three-body initial condition. Sampling its trajectory shows a very close passage by the secondary, supporting the diagnosis. The exact minimum is smaller than a sampled minimum, so the value below is an upper bound on the true closest approach.

In [8]:
sim_ias15 = make_minimal_sim(integrator="ias15")
min_distance = np.full(2, np.inf)
max_com_radius = 0.0

for target in np.linspace(0.0, 5.0 * P_BINARY, 2001)[1:]:
    sim_ias15.integrate(target)
    test = sim_ias15.particles[2]
    com = sim_ias15.com(last=2)
    max_com_radius = max(
        max_com_radius,
        np.sqrt((test.x-com.x)**2 + (test.y-com.y)**2 + (test.z-com.z)**2),
    )
    for j in (0, 1):
        star = sim_ias15.particles[j]
        distance = np.sqrt(
            (test.x-star.x)**2 + (test.y-star.y)**2 + (test.z-star.z)**2
        )
        min_distance[j] = min(min_distance[j], distance)

print(f"IAS15 reached t/P = {sim_ias15.t/P_BINARY:.1f}")
print(f"sampled minimum distance to primary   = {min_distance[0]:.8f}")
print(f"sampled minimum distance to secondary = {min_distance[1]:.8f}")
print(f"maximum COM radius over five periods  = {max_com_radius:.8f}")

IAS15 reached t/P = 5.0
sampled minimum distance to primary   = 1.08467831
sampled minimum distance to secondary = 0.00306103
maximum COM radius over five periods  = 4.28416076


## Recommended changes

### In the C solver

Fail fast rather than spin forever. Before the Stumpff scaling loop, reject non-finite $z$; also cap the number of scaling and bisection iterations. The error should propagate through `reb_simulation_error` and stop integration. Returning NaNs and continuing would hide corrupted dynamics, so an explicit error is preferable.

Conceptually:

```c
if (!isfinite(z)) {
    /* propagate a Kepler convergence/overflow error and abort this step */
}
while (fabs(z) > 0.1 && n < MAX_SCALE) {
    z /= 4.;
    n++;
}
```

### In the experiment

- Treat particles entering a chosen stellar collision/encounter radius according to the intended physics (remove, collide, or hand off to a close-encounter solver).
- Use IAS15 for validation of representative encounters, or a suitable hybrid close-encounter integrator for production if its coordinate assumptions match the experiment.
- Add progress checkpoints and wall-time estimates. Even after fixing the hang, the original request is about 6,000 global steps over 100,000 test particles—roughly 600 million particle-step updates—so it is expected to take minutes rather than seconds.
- Do not interpret a completed coarse fixed-step run as converged: repeat with encounter handling and resolution tests.

## Final diagnosis

The last cell runs forever because one massless particle experiences an unresolved close encounter, the WHFast hyperbolic universal-variable iteration overflows, and `stumpff_cs3` tries forever to scale $-\infty$ into a finite interval. The minimal reproducer needs only three particles and fails on step 44. The 100,000-particle workload makes the original cell expensive, but it is not the reason it never returns.